# RakForest — Single-label Validation Report (Evaluation only)

โน้ตบุ๊กนี้ **ไม่ฝึกโมเดล** โหลด checkpoint Single-label ที่ฝึกเสร็จแล้ว แล้วคำนวณ Validation report แยกรายคลาสในรูปแบบเดียวกับ `classification_report` ของ Multi-label

ใช้ checkpoint `models/rakforest_audio_cnn_singlelabel_retrained_best.pt` ที่สร้างจาก notebook 07 และข้อมูล Validation ปัจจุบัน รายงานจะแสดง Precision, Recall, F1-score, Support รวมถึง Micro/Macro/Weighted/Samples averages


In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score, hamming_loss, classification_report, confusion_matrix
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

PROJECT_ROOT=Path.cwd()
if PROJECT_ROOT.name=='notebooks': PROJECT_ROOT=PROJECT_ROOT.parent
if not (PROJECT_ROOT/'data/processed/processed_manifest.parquet').exists():
    raise FileNotFoundError('เปิด notebook จาก workspace RakForest หรือกำหนด PROJECT_ROOT ให้ถูกต้อง')
MANIFEST_PATH=PROJECT_ROOT/'data/processed/processed_manifest.parquet'
CHECKPOINT_PATH=PROJECT_ROOT/'models/rakforest_audio_cnn_singlelabel_retrained_best.pt'
REPORT_DIR=PROJECT_ROOT/'reports'; ASSET_DIR=REPORT_DIR/'presentation_assets'
REPORT_DIR.mkdir(parents=True,exist_ok=True); ASSET_DIR.mkdir(parents=True,exist_ok=True)
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'ไม่พบ checkpoint ที่ฝึกใหม่: {CHECKPOINT_PATH}; ต้องฝึก Baseline ใน notebook 07 ให้เสร็จก่อน (ไม่ต้องฝึกซ้ำถ้าไฟล์นี้มีแล้ว)')
DEVICE=torch.device('cuda' if torch.cuda.is_available() else ('mps' if hasattr(torch.backends,'mps') and torch.backends.mps.is_available() else 'cpu'))
SAMPLE_RATE=16000; DURATION_SECONDS=4.0; NUM_SAMPLES=64000
N_MELS=64; N_FFT=1024; HOP_LENGTH=320; BATCH_SIZE=32; NUM_WORKERS=0
# Requires a freshly trained 10-class single-label checkpoint.
CLASS_NAMES=['fire','chainsaw','vehicle','human_activity','threat_tactical','gunshot','weather','natural_ambient','silence','other']
CACHE_DIR=PROJECT_ROOT/'data/processed/logmel_cache_v1'; CACHE_DIR.mkdir(parents=True,exist_ok=True)
print('Evaluation only — no training will run')
print('Device:',DEVICE)
print('Checkpoint:',CHECKPOINT_PATH)


### อธิบาย Code cell 1: RakForest — Single-label Validation Report (Evaluation only)

โหลดไลบรารี, กำหนด paths, class order, device และ config ให้ตรงกับ single-label checkpoint





**ตัวแปร/ฟังก์ชันสำคัญ — ความหมายและใช้ทำอะไร:**

- `NUM_WORKERS` — จำนวน process worker ที่ DataLoader ใช้โหลดข้อมูลขนาน
- `hamming_loss` — สัดส่วน label decisions รายคลาสที่ผิด; ต่ำดีกว่า
- `metrics` — พจนานุกรมผล metrics ที่บันทึกหรือแสดง
- `MANIFEST_PATH` — ตำแหน่งไฟล์ processed manifest (Parquet) ที่เป็นสารบัญระหว่างคลิปเสียงกับ label/split
- `REPORT_DIR` — โฟลเดอร์ปลายทางของ reports

**จุดที่ควรจำ:** คำอธิบายนี้อิงโค้ดในเซลล์นี้โดยตรง; ถ้ามี flag/เงื่อนไขบาง branch จะทำงานเฉพาะเมื่อเงื่อนไขนั้นเป็นจริง และ output ที่มีอยู่เดิมอาจมาจากการรันก่อนหน้า ต้องรันเซลล์ใหม่เพื่อให้ตรงกับ config ปัจจุบัน.

## 1. โหลด Validation split และแก้ path ไฟล์เสียงจากตำแหน่งเดิม

In [ ]:
df=pd.read_parquet(MANIFEST_PATH).copy()
if not {'processed_file','target_label','split'}.issubset(df.columns): raise ValueError('Manifest ต้องมี processed_file, target_label และ split')
val_df=df[df.split.eq('val')].reset_index(drop=True)
if val_df.empty: raise ValueError('ไม่พบ Validation rows')

def resolve_audio_path(value):
    original=Path(str(value))
    if original.is_file(): return str(original)
    relocated=PROJECT_ROOT/'data/processed/clips'/original.name
    if relocated.is_file(): return str(relocated)
    raise FileNotFoundError(f'หาไฟล์เสียงไม่พบ: {value}; ลองแล้ว {relocated}')
val_df['processed_file']=val_df.processed_file.map(resolve_audio_path)
unknown=set(val_df.target_label)-set(CLASS_NAMES)
if unknown: raise ValueError(f'คลาสใน manifest ไม่รู้จัก: {sorted(unknown)}')
print('Validation clips:',f'{len(val_df):,}')
print('Validation class support:')
display(val_df.target_label.value_counts().reindex(CLASS_NAMES,fill_value=0).rename('support').to_frame())


### อธิบาย Code cell 2: 1. โหลด Validation split และแก้ path ไฟล์เสียงจากตำแหน่งเดิม

อ่าน manifest แล้วกรองเฉพาะ validation rows; ตรวจคอลัมน์และ resolve paths ของ processed clips; แสดง support ต่อคลาสเพื่อให้เห็นข้อมูลที่ใช้รายงาน





**ตัวแปร/ฟังก์ชันสำคัญ — ความหมายและใช้ทำอะไร:**

- `MANIFEST_PATH` — ตำแหน่งไฟล์ processed manifest (Parquet) ที่เป็นสารบัญระหว่างคลิปเสียงกับ label/split
- `df` — DataFrame ที่โหลดจาก manifest หรือ inventory ในเซลล์นี้; ใช้เป็นตารางฐานสำหรับ filter/validate

**จุดที่ควรจำ:** คำอธิบายนี้อิงโค้ดในเซลล์นี้โดยตรง; ถ้ามี flag/เงื่อนไขบาง branch จะทำงานเฉพาะเมื่อเงื่อนไขนั้นเป็นจริง และ output ที่มีอยู่เดิมอาจมาจากการรันก่อนหน้า ต้องรันเซลล์ใหม่เพื่อให้ตรงกับ config ปัจจุบัน.

In [ ]:
def load_wave(path):
    y,sr=sf.read(str(path),dtype='float32',always_2d=False)
    if y.ndim==2: y=y.mean(axis=1)
    y=np.asarray(y,dtype=np.float32)
    if sr!=SAMPLE_RATE: y=librosa.resample(y,orig_sr=sr,target_sr=SAMPLE_RATE)
    y-=float(y.mean())
    if len(y)<NUM_SAMPLES: y=np.pad(y,(0,NUM_SAMPLES-len(y)))
    else: y=y[:NUM_SAMPLES]
    return y.astype(np.float32)

def to_logmel(y):
    mel=librosa.feature.melspectrogram(y=y,sr=SAMPLE_RATE,n_fft=N_FFT,hop_length=HOP_LENGTH,n_mels=N_MELS,power=2.0)
    logmel=librosa.power_to_db(mel,ref=np.max)
    return ((logmel-logmel.mean())/(logmel.std()+1e-6)).astype(np.float32)

def feature_cache_path(audio_path):
    path=Path(audio_path); stat=path.stat()
    signature=f'{path.resolve()}|{stat.st_size}|{stat.st_mtime_ns}|{SAMPLE_RATE}|{N_MELS}|{N_FFT}|{HOP_LENGTH}'
    return CACHE_DIR/(hashlib.sha1(signature.encode()).hexdigest()+'.npy')

class ValidationDataset(Dataset):
    def __init__(self,frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,index):
        row=self.frame.iloc[index]; cp=feature_cache_path(row.processed_file)
        if cp.exists(): x=np.load(cp).astype(np.float32)
        else:
            x=to_logmel(load_wave(row.processed_file))
            tmp=cp.with_name(cp.stem+'.tmp.npy'); np.save(tmp,x.astype(np.float16)); tmp.replace(cp)
        return torch.from_numpy(x).unsqueeze(0),CLASS_NAMES.index(row.target_label)

val_loader=DataLoader(ValidationDataset(val_df),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'))

class ConvBlock(nn.Module):
    def __init__(self,in_channels,out_channels):
        super().__init__(); self.block=nn.Sequential(nn.Conv2d(in_channels,out_channels,3,padding=1,bias=False),nn.BatchNorm2d(out_channels),nn.ReLU(inplace=True),nn.MaxPool2d(2))
    def forward(self,x): return self.block(x)
class AudioCNN(nn.Module):
    def __init__(self,num_classes):
        super().__init__(); self.features=nn.Sequential(ConvBlock(1,16),ConvBlock(16,32),ConvBlock(32,64),ConvBlock(64,128),nn.AdaptiveAvgPool2d((1,1))); self.classifier=nn.Sequential(nn.Flatten(),nn.Dropout(.35),nn.Linear(128,num_classes))
    def forward(self,x): return self.classifier(self.features(x))


### อธิบาย Code cell 3: 1. โหลด Validation split และแก้ path ไฟล์เสียงจากตำแหน่งเดิม

อ่าน validation clips, ทำ preprocessing/log-Mel เหมือนตอนฝึก และเตรียม Dataset/architecture สำหรับโหลด checkpoint





**ตัวแปร/ฟังก์ชันสำคัญ — ความหมายและใช้ทำอะไร:**

- `audio_path` — path ปลายทางของเสียงที่ export หรือไฟล์ที่ inference กำลังอ่าน
- `NUM_WORKERS` — จำนวน process worker ที่ DataLoader ใช้โหลดข้อมูลขนาน
- `features` — ใน AudioCNN/CRNN คือ convolutional feature extractor; ใน preprocessing คือ acoustic/spectrogram features ตามบริบท
- `classifier` — linear layer แปลง learned representation เป็น score หนึ่งค่าต่อคลาส
- `row` — record ปัจจุบันจาก DataFrame/manifest; คอลัมน์ที่อ่านกำหนดว่าเป็น metadata/label ของไฟล์ใด

**จุดที่ควรจำ:** คำอธิบายนี้อิงโค้ดในเซลล์นี้โดยตรง; ถ้ามี flag/เงื่อนไขบาง branch จะทำงานเฉพาะเมื่อเงื่อนไขนั้นเป็นจริง และ output ที่มีอยู่เดิมอาจมาจากการรันก่อนหน้า ต้องรันเซลล์ใหม่เพื่อให้ตรงกับ config ปัจจุบัน.

## 2. โหลด checkpoint และประเมิน Validation

Cell นี้ทำ inference หนึ่งรอบบน Validation; ไม่มี optimizer หรือ training loop

In [ ]:
checkpoint=torch.load(CHECKPOINT_PATH,map_location=DEVICE,weights_only=True)
if checkpoint.get('class_names')!=CLASS_NAMES: raise ValueError('ลำดับคลาสใน checkpoint ไม่ตรงกับ Validation report')
model=AudioCNN(len(CLASS_NAMES)).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict']); model.eval()
true_batches=[]; pred_batches=[]; prob_batches=[]
with torch.no_grad():
    for xb,yb in tqdm(val_loader,desc='Evaluate single-label validation'):
        probs=torch.softmax(model(xb.to(DEVICE)),dim=1).cpu().numpy()
        true_batches.append(np.asarray(yb)); prob_batches.append(probs); pred_batches.append(probs.argmax(axis=1))
y_true=np.concatenate(true_batches); y_prob=np.concatenate(prob_batches); y_pred=np.concatenate(pred_batches)
y_true_onehot=np.eye(len(CLASS_NAMES),dtype=np.int32)[y_true]
y_pred_onehot=np.eye(len(CLASS_NAMES),dtype=np.int32)[y_pred]
macro_f1=float(f1_score(y_true,y_pred,average='macro',zero_division=0))
accuracy=float(accuracy_score(y_true,y_pred))
micro_f1=float(f1_score(y_true_onehot,y_pred_onehot,average='micro',zero_division=0))
exact_match=float(np.mean(np.all(y_true_onehot==y_pred_onehot,axis=1)))
subset_hamming=float(hamming_loss(y_true_onehot,y_pred_onehot))
print(f"Validation macro-F1: {macro_f1:.4f} | micro-F1: {micro_f1:.4f} | exact match: {exact_match:.4f} | accuracy: {accuracy:.4f} | Hamming loss: {subset_hamming:.4f}")
report_text=classification_report(y_true_onehot,y_pred_onehot,target_names=CLASS_NAMES,digits=3,zero_division=0)
print(report_text)
report_path=REPORT_DIR/'singlelabel_validation_classification_report.txt'
report_path.write_text(report_text,encoding='utf-8')
print('Saved:',report_path)


### อธิบาย Code cell 4: 2. โหลด checkpoint และประเมิน Validation

โหลด checkpoint, ตรวจ class order/config, สร้าง AudioCNN และทำนาย validation ด้วย softmax/argmax; ไม่ได้ train โมเดลใหม่





**ตัวแปร/ฟังก์ชันสำคัญ — ความหมายและใช้ทำอะไร:**

- `macro_f1` — F1 ที่คำนวณแยกคลาสและเฉลี่ย ไม่ถ่วงตามขนาดคลาส
- `micro_f1` — F1 จาก TP/FP/FN รวมทุกคลาส
- `accuracy` — สัดส่วนคำทำนาย class เดียวที่ถูก
- `exact_match` — สัดส่วนคำตอบ one-hot/multi-hot ที่ตรงครบ
- `hamming_loss` — สัดส่วน label decisions รายคลาสที่ผิด; ต่ำดีกว่า
- `true_batches` — list เก็บ labels จริงจาก validation batches
- `probs` — คะแนนทำนายหลัง softmax/sigmoid
- `y_true_onehot` — labels จริงแปลงเป็น one-hot เพื่อเทียบแบบ multi-label-format
- `y_pred_onehot` — คำทำนาย class index แปลงเป็น one-hot
- `subset_hamming` — สัดส่วนองค์ประกอบ one-hot ที่ทายผิด
- `report_text` — classification report แบบข้อความ
- `report_path` — ตำแหน่งบันทึกรายงาน classification
- `xb` — batch input features; b หมายถึง batch dimension
- `yb` — batch target labels คู่กับ input batch
- `y_prob` — คะแนน sigmoid ต่อคลาสก่อนใช้ threshold
- `model` — โมเดล CNN/CRNN ปัจจุบันที่รับ log-Mel batch และคืน logits หนึ่งช่องต่อคลาส
- `REPORT_DIR` — โฟลเดอร์ปลายทางของ reports

**จุดที่ควรจำ:** คำอธิบายนี้อิงโค้ดในเซลล์นี้โดยตรง; ถ้ามี flag/เงื่อนไขบาง branch จะทำงานเฉพาะเมื่อเงื่อนไขนั้นเป็นจริง และ output ที่มีอยู่เดิมอาจมาจากการรันก่อนหน้า ต้องรันเซลล์ใหม่เพื่อให้ตรงกับ config ปัจจุบัน.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
precision,recall,f1,support=precision_recall_fscore_support(y_true,y_pred,labels=np.arange(len(CLASS_NAMES)),average=None,zero_division=0)
report_table=pd.DataFrame({'class':CLASS_NAMES,'precision':precision,'recall':recall,'f1-score':f1,'support':support.astype(int)})
display(report_table.round(3))
table_path=REPORT_DIR/'singlelabel_validation_classification_report.csv'
report_table.to_csv(table_path,index=False)

cm=confusion_matrix(y_true,y_pred,labels=np.arange(len(CLASS_NAMES)))
cm_path=REPORT_DIR/'singlelabel_validation_confusion_matrix.csv'
pd.DataFrame(cm,index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(cm_path)
fig,ax=plt.subplots(figsize=(9,7)); im=ax.imshow(cm,cmap='Blues')
ax.set_xticks(np.arange(len(CLASS_NAMES)),CLASS_NAMES,rotation=45,ha='right'); ax.set_yticks(np.arange(len(CLASS_NAMES)),CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Single-label Baseline — Validation confusion matrix')
fig.colorbar(im,ax=ax,label='Clips'); fig.tight_layout()
image_path=ASSET_DIR/'rakforest_singlelabel_validation_confusion_matrix.png'
fig.savefig(image_path,dpi=220,bbox_inches='tight'); plt.show()
print('Saved:',table_path,cm_path,image_path)
print('Checkpoint best epoch:',checkpoint.get('best_epoch'),'| stored best validation Macro-F1:',checkpoint.get('best_val_macro_f1'))


### อธิบาย Code cell 5: 2. โหลด checkpoint และประเมิน Validation

โหลดไลบรารี, กำหนด paths, class order, device และ config ให้ตรงกับ single-label checkpoint





**ตัวแปร/ฟังก์ชันสำคัญ — ความหมายและใช้ทำอะไร:**

- `precision` — สัดส่วนคำทำนาย positive ที่ถูกต้อง
- `recall` — สัดส่วน positive จริงที่โมเดลตรวจพบ
- `f1` — ค่าเฉลี่ยสมดุลระหว่าง precision และ recall
- `metrics` — พจนานุกรมผล metrics ที่บันทึกหรือแสดง
- `report_table` — DataFrame ที่รวม precision/recall/F1/support ต่อคลาส
- `table_path` — path ของ per-class report CSV
- `cm_path` — path ของ confusion matrix CSV
- `image_path` — path ของกราฟ/ภาพที่ export
- `fig` — วัตถุ figure ของ Matplotlib ที่รวมกราฟในภาพเดียว
- `REPORT_DIR` — โฟลเดอร์ปลายทางของ reports

**จุดที่ควรจำ:** คำอธิบายนี้อิงโค้ดในเซลล์นี้โดยตรง; ถ้ามี flag/เงื่อนไขบาง branch จะทำงานเฉพาะเมื่อเงื่อนไขนั้นเป็นจริง และ output ที่มีอยู่เดิมอาจมาจากการรันก่อนหน้า ต้องรันเซลล์ใหม่เพื่อให้ตรงกับ config ปัจจุบัน.

## 3. Error Analysis — False Positive / False Negative และตัวอย่างที่ทายผิด

สรุปจำนวน FP/FN ต่อคลาสจาก Validation และเปิดเสียงตัวอย่างที่ Single ทายผิดให้ฟังได้ ใช้เพื่อหาประเภทข้อผิดพลาด ไม่ได้เลือก threshold หรือโมเดลจาก Test


In [ ]:
from IPython.display import Audio, display, Markdown

error_rows=val_df.copy()
error_rows['true_label']=[CLASS_NAMES[i] for i in y_true]
error_rows['predicted_label']=[CLASS_NAMES[i] for i in y_pred]
error_rows['prediction_confidence']=y_prob.max(axis=1)
error_rows['true_class_score']=y_prob[np.arange(len(y_true)),y_true]
error_rows['predicted_class_score']=y_prob[np.arange(len(y_pred)),y_pred]
errors=error_rows[error_rows.true_label.ne(error_rows.predicted_label)].copy()
errors=errors.sort_values('prediction_confidence',ascending=False)
print(f'Misclassified validation clips: {len(errors):,}/{len(val_df):,} ({len(errors)/max(len(val_df),1):.1%})')

error_summary=[]
for idx,name in enumerate(CLASS_NAMES):
    error_summary.append({
        'class':name,
        'support':int(np.sum(y_true==idx)),
        'false_positive':int(np.sum((y_pred==idx)&(y_true!=idx))),
        'false_negative':int(np.sum((y_true==idx)&(y_pred!=idx))),
        'correct':int(np.sum((y_true==idx)&(y_pred==idx))),
    })
error_summary_df=pd.DataFrame(error_summary)
display(error_summary_df)
error_summary_path=REPORT_DIR/'singlelabel_validation_error_analysis_by_class.csv'
error_summary_df.to_csv(error_summary_path,index=False)
error_path=REPORT_DIR/'singlelabel_validation_misclassified_clips.csv'
errors[['dataset','processed_file','true_label','predicted_label','prediction_confidence','true_class_score','predicted_class_score']].to_csv(error_path,index=False)
print('Saved:',error_summary_path,error_path)

ERROR_PREVIEW_LIMIT=5
for row in errors.head(ERROR_PREVIEW_LIMIT).itertuples():
    display(Markdown(f'### {row.dataset} | true: **{row.true_label}** → predicted: **{row.predicted_label}** | confidence: {row.prediction_confidence:.3f}'))
    display(Audio(load_wave(row.processed_file),rate=SAMPLE_RATE))
if errors.empty: print('No misclassified validation clips to preview.')


### อธิบาย Code cell 6: 3. Error Analysis — False Positive / False Negative และตัวอย่างที่ทายผิด

โหลดไลบรารี, กำหนด paths, class order, device และ config ให้ตรงกับ single-label checkpoint





**ตัวแปร/ฟังก์ชันสำคัญ — ความหมายและใช้ทำอะไร:**

- `ERROR_PREVIEW_LIMIT` — จำนวนตัวอย่าง error ที่แสดงให้ฟัง
- `error_summary_df` — ตารางสรุป error ต่อคลาส/ประเภทความผิดพลาด
- `error_summary_path` — ตำแหน่ง CSV error summary
- `error_path` — ตำแหน่ง CSV รายการคลิปที่ทายผิด
- `error_rows` — ตาราง validation/test ที่ทายผิดเพื่อเปิดฟังตรวจ
- `error_summary` — จำนวน error สรุปตามคลาสหรือประเภทความผิด
- `row` — record ปัจจุบันจาก DataFrame/manifest; คอลัมน์ที่อ่านกำหนดว่าเป็น metadata/label ของไฟล์ใด
- `y_prob` — คะแนน sigmoid ต่อคลาสก่อนใช้ threshold
- `REPORT_DIR` — โฟลเดอร์ปลายทางของ reports

**จุดที่ควรจำ:** คำอธิบายนี้อิงโค้ดในเซลล์นี้โดยตรง; ถ้ามี flag/เงื่อนไขบาง branch จะทำงานเฉพาะเมื่อเงื่อนไขนั้นเป็นจริง และ output ที่มีอยู่เดิมอาจมาจากการรันก่อนหน้า ต้องรันเซลล์ใหม่เพื่อให้ตรงกับ config ปัจจุบัน.

## 4. หมายเหตุ

- Notebook นี้ประเมิน checkpoint ที่มีอยู่ ไม่ฝึกโมเดลซ้ำ
- หาก `02_eda_cleaning.ipynb` เพิ่งสร้าง manifest/split ชุดใหม่ ให้ใช้ checkpoint ที่ฝึกหลังจากนั้น (ชื่อ `rakforest_audio_cnn_singlelabel_retrained_best.pt`)
- ผลทั้งหมดในไฟล์นี้เป็น Validation ไม่ใช่ Test
